In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from mlx_vlm import load, generate

# Loading the optimal E4B model for your M3 Pro
model, tokenizer = load("mlx-community/gemma-4-e4b-it-4bit")

/opt/miniconda3/envs/gaai/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 114130.72it/s]


In [3]:
from pathlib import Path
for label, folder, ext in [('Post-GDPR', '../data/post_gdpr', '*.xml')]:
    post_gdpr_files = list(Path(folder).rglob(ext))
    print(f'{label}: {len(post_gdpr_files)} {ext} files found')
for label, folder, ext in [('Pre-GDPR', '../data/pre_gdpr', '*.md')]:
    pre_gdpr_files = list(Path(folder).rglob(ext))
    print(f'{label}: {len(pre_gdpr_files)} {ext} files found')

files = post_gdpr_files + pre_gdpr_files

Post-GDPR: 150 *.xml files found
Pre-GDPR: 200 *.md files found


In [4]:
files

[PosixPath('../data/post_gdpr/data/GoPPC-150/88.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/63.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/77.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/76.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/62.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/89.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/149.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/48.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/74.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/60.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/61.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/75.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/49.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/148.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/71.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/65.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/59.xml'),
 PosixPath('../data/post_gdpr/data/GoPPC-150/58.xml'),
 PosixPa

In [5]:
from utils import xml2txt, md2txt, pdf2txt

In [6]:
files[0]

PosixPath('../data/post_gdpr/data/GoPPC-150/88.xml')

In [7]:
str(files[0]).endswith('.xml')

True

In [8]:
def preprocess_file(file_path: str):
    if file_path.endswith('.xml'):

        text = xml2txt(file_path)
    elif file_path.endswith('.md'):
        text = md2txt(file_path)
    elif file_path.endswith('.pdf'):
        text = pdf2txt(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_path}")
    return text

txt_files = [preprocess_file(str(file)) for file in files]


In [9]:
import pprint

In [10]:
pprint.pp(txt_files[0])

('Welcome to Patreon!\n'
 'Patreon is a platform where patrons can support and engage with creators. '
 'This \xa0updated Privacy Policy applies to patrons, creators, and all users '
 'of our platform, and is part of our Terms of Use.\n'
 'Patreon is a global company. By using our platform, you agree that your '
 'personal information that you provide directly to us, or that we collect '
 'through your use of the platform, may be transferred to and stored in the '
 'United States and handled as described in this Policy.\n'
 'Information You Provide Through Your Account\n'
 'This is information that you provide to us through text fields, such as your '
 'name, payment information and benefits. The information we collect differs '
 'depending on if you make an account, become a patron, or become a creator.\n'
 'First and Last Name\n'
 'Email Address\n'
 'Username\n'
 'Password\n'
 'State and Country of Residence\n'
 'You may also sign up using a Facebook or Google account. We will ask '


In [11]:
pprint.pp(txt_files[-1])

('The following text is extracted and transformed from the a4academics.com '
 'privacy policy that was archived on 2019-07-19. Please check the original '
 'snapshot on the Wayback Machine for the most accurate reproduction.\n'
 'Privacy Policy & Terms Of Service\n'
 '\n'
 ' \n'
 'Privacy Policy\n'
 '\n'
 'To use our site, registration is not mandatory. But to get latest and '
 'interesting updates, we suggest users to subscribe to our weekly newsletter. '
 'We assure you that we will NOT trade, rent or share your emails with any '
 'other third parties. We had implemented enough security mechanisms to '
 'protect your personal information, transaction information and data stored '
 'on our Site. We will use your email, if we need to notify you about '
 'important policy updates or to get feedback on topics you wish to read. If '
 'you feel that our email services is not adding value to you, you can '
 'unsubscribe from our newsletter, provided at the bottom of each email. \n'
 'Non-pe

## Importing metadata

In [12]:
import json
from dataclass import ArticlePolicy, KeywordNode

In [13]:
with open('../metadata/keyword_nodes.json', 'r') as f:
    kw_nodes= [KeywordNode(**k) for k in json.load(f)]

In [14]:
pprint.pp(kw_nodes[0])

KeywordNode(node=1,
            keywords=['personal data protection',
                      'fundamental rights',
                      'free movement of data',
                      'natural persons',
                      'regulation scope',
                      'eu 2016/679',
                      'data subject rights',
                      'protection of individuals'],
            chapter=1,
            section=1)


In [15]:
with open('../metadata/article_policies.json', 'r') as f:
    article_policies = [ArticlePolicy(**p) for p in json.load(f)]

In [16]:
pprint.pp(article_policies[0])

ArticlePolicy(number=2,
              title='Material scope',
              priority='p1',
              hil='hil-cond',
              action='Run first. Auto-confirm company is in scope. If any '
                     'exemption is claimed (household, law enforcement), '
                     'quarantine chunk and halt pipeline — route to human '
                     'before proceeding.',
              chapter='Ch.1 – General provisions',
              section=None)


## Langgraph

In [17]:
from typing import Any, TypedDict

In [18]:
from dataclass import Document

In [ ]:

class WorkflowState(TypedDict, total=False):
    # Inputs
    document_paths: list[str]
    chunk_chars: int
    overlap_chars: int

    # Mapping
    mapping_min_chars: int
    mapping_max_chunks: int
    mapping_candidate_chunks: int

    # Derived/working
    documents: list[Document]
    full_text: str
    # article_policies: list[ArticlePolicy]
    relevant_articles: list[ArticlePolicy]
    scope: dict[str, Any]
    chunks: list[dict[str, Any]]
    article_store: dict[int, list[dict[str, Any]]]

    # Dummy-RAG
    rag_articles: dict[int, dict[str, Any]]  # article_number -> {text, summary, used}

    # Checks + outputs
    findings: list[dict[str, Any]]
    hil_queue: list[dict[str, Any]]
    report: dict[str, Any]



In [20]:
eg_workflow_state = WorkflowState(
    document_paths= [str(file) for file in files[:1]],
    chunk_chars= 1200,
    overlap_chars= 200,
)

In [21]:


def _dummy_rag_fetch_article(article_number: int) -> dict[str, Any]:
    """Dummy RAG: pretend we retrieved the GDPR article text.

    - Returns `text` for small articles
    - Returns `summary` for large articles (simulated)

    Replace this with a real retriever later (vector DB, files, API, etc.).
    """
    # Simulate size by a deterministic rule: every 3rd article is "large".
    is_large = (article_number % 3) == 0
    if is_large:
        return {
            "used": "summary",
            "text": None,
            "summary": (
                f"Compressed summary for GDPR Article {article_number}. "
                "Key obligations are condensed for checker input."
            ),
        }
    return {
        "used": "text",
        "text": (
            f"Full retrieved text for GDPR Article {article_number}. "
            "This is dummy placeholder content for the workflow."
        ),
        "summary": None,
    }


In [22]:
from langgraph.graph import END, START, StateGraph

In [23]:

from utils import _normalize_text

In [24]:
def ingestion_node(state: WorkflowState) -> WorkflowState:
        paths = state.get("document_paths") or []
        docs = [preprocess_file(p) for p in paths] # todo : here provision for multiple documents with policies for the same company, can add multiple documents with policies for the same company, 
        full_text = _normalize_text("\n\n".join(d for d in docs))
        return {"documents": docs, "full_text": full_text}

In [25]:
eg_workflow_state

{'document_paths': ['../data/post_gdpr/data/GoPPC-150/88.xml'],
 'chunk_chars': 1200,
 'overlap_chars': 200}

In [26]:
op = ingestion_node(eg_workflow_state)
eg_workflow_state['full_text'] = op['full_text']
eg_workflow_state['documents'] = op['documents']
eg_workflow_state

{'document_paths': ['../data/post_gdpr/data/GoPPC-150/88.xml'],
 'chunk_chars': 1200,
 'overlap_chars': 200,
 'full_text': 'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.\nInformation You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name\nEmail Address\nUsername\nPassword\nState and Country of Residence\nYou may also sign

In [27]:
eg_workflow_state

{'document_paths': ['../data/post_gdpr/data/GoPPC-150/88.xml'],
 'chunk_chars': 1200,
 'overlap_chars': 200,
 'full_text': 'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.\nInformation You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name\nEmail Address\nUsername\nPassword\nState and Country of Residence\nYou may also sign

In [28]:

def filter_nonskip_articles(articles: Iterable[ArticlePolicy]) -> Iterable[ArticlePolicy]:
    for a in articles:
        if a.priority != "skip":
            yield a

In [29]:
def load_reference_node(state: WorkflowState) -> WorkflowState:
    with open('../metadata/article_policies.json', 'r') as f:
        article_policies = [ArticlePolicy(**p) for p in json.load(f)]

    relevant = list(filter_nonskip_articles(article_policies))
    return { "relevant_articles": relevant}



In [30]:
eg_workflow_state['relevant_articles'] = load_reference_node(eg_workflow_state)['relevant_articles']

In [31]:
from local_model import _build_json_llm_agent

In [32]:
local = False

In [33]:
# !pip install pydantic
# !pip install dotenv

In [34]:
scope_agent = _build_json_llm_agent(required_keys=["applies", "reasons", "evidence", "hil_required"], local=local)
# check_agent = _build_json_llm_agent(
#         required_keys=["status", "gaps", "evidence", "risk", "needs_human_review", "notes"], local=local
#     )

In [35]:
# !pip install google-genai

In [36]:
from dataclass import Prompts


In [37]:


DEFAULT_PROMPTS = Prompts(
    scope_gate_system=(
        "You are a GDPR compliance analyst.\n"
        "Decide whether GDPR likely applies based on the provided policy/company context.\n"
        "Focus only on GDPR Art.2 (material scope) and Art.3 (territorial scope).\n"
        "If scope is ambiguous or any exemption is claimed, escalate to human-in-loop.\n"
        "Return concise evidence quotes from the text when possible."
    ),
    scope_gate_user=(
        "Analyze the following policy text.\n\n"
        "Return JSON with keys:\n"
        '- applies: one of ["yes","no","unclear"]\n'
        "- reasons: short bullets\n"
        "- evidence: array of short quotes\n"
        "- hil_required: boolean\n\n"
        "POLICY TEXT:\n"
        "{text}"
    ),
    article_check_system=(
        "You are a GDPR compliance checker.\n"
        "Given a specific GDPR article (number + title + policy intent) and the relevant policy chunks,\n"
        "assess whether the policy appears to meet the obligation.\n"
        "If the obligation cannot be verified from policy text alone (operational/contractual facts), mark as needs_human_review.\n"
        "Be strict: absence of explicit language should be treated as a gap."
    ),
    article_check_user=(
        "Check GDPR Article {article_number}: {article_title}\n"
        "Priority: {priority}. Human-in-loop hint: {hil_flag}.\n"
        "Expected agent action/rationale: {action}\n\n"
        "Return JSON with keys:\n"
        '- status: one of ["pass","partial","fail","unknown"]\n'
        "- gaps: array of missing requirements (strings)\n"
        "- evidence: array of short quotes from chunks\n"
        "- risk: one of [\"low\",\"medium\",\"high\",\"critical\"]\n"
        "- needs_human_review: boolean\n"
        "- notes: short free-text\n\n"
        "RELEVANT CHUNKS:\n"
        "{chunks}"
    ),
)



In [38]:
%reload_ext autoreload

In [39]:
from utils import _load_scope_articles_text

In [40]:
import json

In [41]:
scope_ref_text = _load_scope_articles_text("metadata/scope_gate.json")

Error loading scope articles text: [Errno 2] No such file or directory: 'metadata/scope_gate.json'


In [42]:
scope_ref_text

'\n        "articles": [\n    {\n      "number": 2,\n      "title": "Material scope",\n      "text": "This Regulation applies to the processing of personal data wholly or partly by automated means and to the processing other than by automated means of personal data which form part of a filing system or are intended to form part of a filing system.\n\nThis Regulation does not apply to the processing of personal data:\n- in the course of an activity which falls outside the scope of Union law;\n- by the Member States when carrying out activities which fall within the scope of Chapter 2 of Title V of the TEU;\n- by a natural person in the course of a purely personal or household activity;\n- by competent authorities for the purposes of the prevention, investigation, detection or prosecution of criminal offences or the execution of criminal penalties, including the safeguarding against and the prevention of threats to public security.\n\nFor the processing of personal data by the Union inst

In [43]:
prompts = DEFAULT_PROMPTS

In [44]:
def scope_gate_node(state: WorkflowState) -> WorkflowState:
        full_text = state.get("full_text") or ""
        user_prompt = (
            "Use the following GDPR reference text for scope assessment.\n\n"
            f"{scope_ref_text}\n\n"
            "Now analyze this company policy text:\n\n"
            + (full_text[:1200])
        )
        scope = scope_agent(prompts.scope_gate_system, user_prompt) or {
            "applies": "unclear",
            "reasons": [],
            "evidence": [],
            "hil_required": True,
        }
        return {"scope": scope}

In [45]:
prompts

Prompts(scope_gate_system='You are a GDPR compliance analyst.\nDecide whether GDPR likely applies based on the provided policy/company context.\nFocus only on GDPR Art.2 (material scope) and Art.3 (territorial scope).\nIf scope is ambiguous or any exemption is claimed, escalate to human-in-loop.\nReturn concise evidence quotes from the text when possible.', scope_gate_user='Analyze the following policy text.\n\nReturn JSON with keys:\n- applies: one of ["yes","no","unclear"]\n- reasons: short bullets\n- evidence: array of short quotes\n- hil_required: boolean\n\nPOLICY TEXT:\n{text}', article_check_system='You are a GDPR compliance checker.\nGiven a specific GDPR article (number + title + policy intent) and the relevant policy chunks,\nassess whether the policy appears to meet the obligation.\nIf the obligation cannot be verified from policy text alone (operational/contractual facts), mark as needs_human_review.\nBe strict: absence of explicit language should be treated as a gap.', art

In [46]:
ejbf

NameError: name 'ejbf' is not defined

In [47]:
response = scope_gate_node(eg_workflow_state)

In [48]:
response

{'scope': {'applies': True,
  'reasons': ['Material Scope (Art. 2): The company processes personal data (e.g., "name, payment information, First and Last Name, Email Address, Username, Password, State and Country of Residence") partly by automated means (e.g., "through your use of the platform", "sign up using a Facebook or Google account"). No exemptions under Article 2 apply.',
   'Territorial Scope (Art. 3): As a "global company" offering a "platform" to "all users of our platform" and collecting "State and Country of Residence", it is highly probable that the company offers goods or services to data subjects who are in the Union (Art. 3(2)(a)). Furthermore, by collecting "information... through your use of the platform", it is likely monitoring the behavior of data subjects who are in the Union (Art. 3(2)(b)).'],
  'evidence': ['"Patreon is a global company."',
   '"This updated Privacy Policy applies to patrons, creators, and all users of our platform"',
   '"your personal informa

In [49]:
pprint.pp(response)

{'scope': {'applies': True,
           'reasons': ['Material Scope (Art. 2): The company processes '
                       'personal data (e.g., "name, payment information, First '
                       'and Last Name, Email Address, Username, Password, '
                       'State and Country of Residence") partly by automated '
                       'means (e.g., "through your use of the platform", "sign '
                       'up using a Facebook or Google account"). No exemptions '
                       'under Article 2 apply.',
                       'Territorial Scope (Art. 3): As a "global company" '
                       'offering a "platform" to "all users of our platform" '
                       'and collecting "State and Country of Residence", it is '
                       'highly probable that the company offers goods or '
                       'services to data subjects who are in the Union (Art. '
                       '3(2)(a)). Furthermore, by collecting "

In [50]:
eg_workflow_state['scope'] = response['scope']

In [51]:
eg_workflow_state

{'document_paths': ['../data/post_gdpr/data/GoPPC-150/88.xml'],
 'chunk_chars': 1200,
 'overlap_chars': 200,
 'full_text': 'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.\nInformation You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name\nEmail Address\nUsername\nPassword\nState and Country of Residence\nYou may also sign

In [52]:
import re

In [59]:

def _chunk_text(text: str, *, chunk_chars: int = 1200, overlap_chars: int = 200) -> list[dict[str, Any]]:
    """Structure-aware chunking (headings + paragraphs) with fallback overlap splitting.

    Strategy:
    - First, try to split into coherent *sections* using heading detection and blank-line paragraphing.
    - If a section is larger than `chunk_chars`, sub-split that section into overlapping windows.

    Returned offsets are in the normalized text coordinate space (post `_normalize_text`).
    """
    text = _normalize_text(text)
    if not text:
        return []
    if chunk_chars <= 0:
        chunk_chars = 1200
    if overlap_chars < 0:
        overlap_chars = 0

    def is_heading(line: str) -> bool:
        s = (line or "").strip()
        if not s:
            return False
        if s.startswith("#"):  # markdown heading
            return True
        if len(s) > 120:
            return False
        if re.match(r"^\d+(\.\d+){0,4}\s+.+$", s):  # 1. / 1.2.3 Title
            return True
        if re.match(r"^[A-Z0-9][A-Z0-9 \-]{6,}$", s) and any(c.isalpha() for c in s):
            # ALL CAPS heading-ish
            return True
        if re.match(r"^([A-Z][a-z]+)(\s+[A-Z][a-z]+){0,8}$", s) and len(s.split()) <= 8:
            # Title Case short line
            return True
        if s.endswith(":") and len(s.split()) <= 8:  # "Retention:" style
            return True
        return False

    # Build section spans with offsets in normalized text
    line_iter = list(re.finditer(r".*(?:\n|$)", text))
    sections: list[dict[str, Any]] = []

    cur_start = 0
    cur_heading: str | None = None

    def flush(end_pos: int) -> None:
        nonlocal cur_start, cur_heading
        if end_pos <= cur_start:
            return
        sec_text = text[cur_start:end_pos].strip()
        if not sec_text:
            cur_start = end_pos
            cur_heading = None
            return
        sections.append(
            {
                "start": cur_start,
                "end": end_pos,
                "heading": cur_heading,
                "text": sec_text,
            }
        )
        cur_start = end_pos
        cur_heading = None

    for m in line_iter:
        line = m.group(0)
        line_start = m.start()

        if is_heading(line):
            # Start a new section at this heading; flush what came before.
            flush(line_start)
            cur_start = line_start
            cur_heading = line.strip()
            continue

        # Paragraph boundary: two newlines already normalized; if this is a blank line
        if line.strip() == "":
            # keep accumulating; blank line ends paragraph but not necessarily section
            continue

    flush(len(text))

    # If heading detection failed (single section), still do paragraph-aware split
    if len(sections) <= 1:
        paras: list[dict[str, Any]] = []
        for pm in re.finditer(r"(?:[^\n].*?)(?:\n\n|$)", text, flags=re.S):
            p_text = pm.group(0).strip()
            if not p_text:
                continue
            paras.append({"start": pm.start(), "end": pm.end(), "heading": None, "text": p_text})
        if paras:
            sections = paras

    def split_with_overlap(sec: dict[str, Any], *, idx0: int) -> list[dict[str, Any]]:
        start = int(sec["start"])
        end = int(sec["end"])
        sec_text = text[start:end]
        sec_len = len(sec_text)
        if sec_len <= chunk_chars:
            return [
                {
                    "chunk_id": f"c{idx0}",
                    "start": start,
                    "end": end,
                    "text": sec_text.strip(),
                    "heading": sec.get("heading"),
                }
            ]
        step = max(1, chunk_chars - overlap_chars)
        out: list[dict[str, Any]] = []
        j = 0
        while j < sec_len:
            sub_start = start + j
            sub_end = min(end, start + j + chunk_chars)
            sub_text = text[sub_start:sub_end].strip()
            out.append(
                {
                    "chunk_id": f"c{idx0 + len(out)}",
                    "start": sub_start,
                    "end": sub_end,
                    "text": sub_text,
                    "heading": sec.get("heading"),
                }
            )
            if sub_end >= end:
                break
            j += step
        return out

    chunks: list[dict[str, Any]] = []
    idx = 0
    for sec in sections:
        new_chunks = split_with_overlap(sec, idx0=idx)
        chunks.extend(new_chunks)
        idx += len(new_chunks)

    # Final fallback: ensure we always return at least one chunk
    if not chunks:
        chunks = [{"chunk_id": "c0", "start": 0, "end": len(text), "text": text, "heading": None}]
    return chunks


In [60]:
def chunking_node(state: WorkflowState) -> WorkflowState:
        full_text = state.get("full_text") or ""
        chunk_chars = int(state.get("chunk_chars") or 1200)
        overlap_chars = int(state.get("overlap_chars") or 200)
        chunks = _chunk_text(full_text, chunk_chars=chunk_chars, overlap_chars=overlap_chars)
        return {"chunks": chunks}

In [61]:
eg_workflow_state['full_text']

'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.\nInformation You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name\nEmail Address\nUsername\nPassword\nState and Country of Residence\nYou may also sign up using a Facebook or Google account. We will ask permission to access basic information from your Facebook or Google acc

In [62]:
res = chunking_node(eg_workflow_state)

In [63]:
res

{'chunks': [{'chunk_id': 'c0',
   'start': 0,
   'end': 491,
   'text': 'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.',
   'heading': None},
  {'chunk_id': 'c1',
   'start': 491,
   'end': 783,
   'text': 'Information You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name',
   'heading': 'Information You Provide Through 

In [97]:
eg_workflow_state['chunks'] = res['chunks']

In [128]:
mapping_agent = _build_json_llm_agent(required_keys=["article_numbers", "notes"], local=False)


Gemini API model loaded


In [ ]:
def mapping_node(state: WorkflowState) -> WorkflowState:

  # Load pre-filtered keyword nodes (relevant articles only).
        kw_json_path = "../metadata/keyword_nodes.json"
        with open(kw_json_path, "r", encoding="utf-8") as f:
            kw_nodes_raw = json.load(f)

        chunks = state.get("chunks") or []
        relevant_articles = state.get("relevant_articles") or []
        mapping_bundle_size = max(1, int(state.get("mapping_bundle_size") or 2))
        mapping_max_bundles = int(state.get("mapping_max_bundles") or 0)  # 0 => no cap
        print("Mapping bundle size", mapping_bundle_size)
        print("Mapping max bundles", mapping_max_bundles)
        relevant_set = {int(a.number) for a in relevant_articles}

        # Put ALL article keywords once in the system prompt. Keep it minimal.
        keyword_reference = json.dumps(kw_nodes_raw, ensure_ascii=False)
        # print("Keyword reference", keyword_reference)
        mapping_system_prompt = (
            "You are a GDPR mapping agent.\n"
            "Given a small bundle of policy text, return which GDPR articles it is relevant to.\n"
            "Return JSON with keys:\n"
            "- article_numbers: array of integers (GDPR article numbers). Use [] if none.\n"
            "- notes: short string.\n\n"
            "Keyword reference JSON (each item: {node:<article_number>, keywords:[...]}):\n"
            f"{keyword_reference}"
        )

        article_store: dict[int, list[dict[str, Any]]] = {n: [] for n in relevant_set}
        seen_by_art: dict[int, set[str]] = {n: set() for n in relevant_set}

        # print("bundle size", mapping_bundle_size, "Chunks", chunks)
        res_js = []
        bundle_count = 0
        for i in range(0, len(chunks), mapping_bundle_size):
            print("Bundle count", bundle_count)
            if mapping_max_bundles > 0 and bundle_count >= mapping_max_bundles:
                break
            b_chunks = chunks[i : i + mapping_bundle_size]
            if not b_chunks:
                continue

            bundle_text = "\n\n".join((c.get("text") or "").strip() for c in b_chunks if (c.get("text") or "").strip())
            if not bundle_text:
                continue
            # print("Bundle text", bundle_text)
            js = mapping_agent(mapping_system_prompt, bundle_text) or {}
            if local : 
                res_js.append(js)
                bundle_count += 1
                continue
            
            arts = js.get("article_numbers") or []
            if isinstance(arts, (int, str)):
                arts = [arts]

            cleaned: list[int] = []
            for a in arts:
                try:
                    n = int(str(a).strip())
                except Exception:
                    continue
                if n in relevant_set:
                    cleaned.append(n)

            for art_no in cleaned:
                for c in b_chunks:
                    cid = str(c.get("chunk_id"))
                    if cid in seen_by_art[art_no]:
                        continue
                    article_store[art_no].append(c)
                    seen_by_art[art_no].add(cid)

            bundle_count += 1

        if local:
            return res_js
        return {"article_store": article_store}

In [131]:
res_mapping = mapping_node(eg_workflow_state)

Mapping bundle size 2
Mapping max bundles 0


In [132]:
res_mapping

{'article_store': {2: [],
  3: [{'chunk_id': 'c0',
    'start': 0,
    'end': 491,
    'text': 'Welcome to Patreon!\nPatreon is a platform where patrons can support and engage with creators. This \xa0updated Privacy Policy applies to patrons, creators, and all users of our platform, and is part of our Terms of Use.\nPatreon is a global company. By using our platform, you agree that your personal information that you provide directly to us, or that we collect through your use of the platform, may be transferred to and stored in the United States and handled as described in this Policy.',
    'heading': None},
   {'chunk_id': 'c1',
    'start': 491,
    'end': 783,
    'text': 'Information You Provide Through Your Account\nThis is information that you provide to us through text fields, such as your name, payment information and benefits. The information we collect differs depending on if you make an account, become a patron, or become a creator.\nFirst and Last Name',
    'heading': 'Inf

the code keeps crashing look at demo.ipynb in code folder 
Is it running on GPU? the model?
is it too much for mac m3 pro to run the model, estimate the compute and memory needed and lmk if am compute or memory bound

and also suggest what to do, is there any way to host the model somewhere else? or smthg?